In [35]:
import os
import sys
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

Running with 8 CPU cores | pyarrow 24.0.0


Variables

In [ ]:
%run 0_variables.ipynb

Feature selection

In [45]:
_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
feature_data_unique = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_feature_data_unique_{_stem}.parquet")
)

features = pd.read_parquet(os.environ["FEATURES_PROCESSED_PATH"])

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
future_prediction_targets_agg = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_targets_agg_{_stem}.parquet")
)

def feature_data_selection(
    feature_data,
    features,
    future_prediction_targets_agg,
    subsample_start: pd.Timestamp,
    subsample_end: pd.Timestamp,
    subsample_amount: int,
):
    # Number of equally-spaced k values to evaluate between 1 and n_available_features — tune this
    N_K_VALUES = 3
    # Number of equally-spaced horizons to evaluate — e.g. 10 tests every ~10th horizon; None = all
    N_HORIZON_STEPS = 5
    N_CV_SPLITS = 5   # time-series walk-forward folds
    # Thread count — each thread holds one X_h copy in the shared process heap (no worker spawning)
    N_PARALLEL_JOBS = min(_NCPU, 8)

    # Lightweight LightGBM for CV speed — not production params
    _LGBM_CV_PARAMS = {
        "objective": "regression_l1",
        "metric": "mae",
        "n_estimators": 500,       # high ceiling; early stopping exits well before this
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_child_samples": 20,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "n_iter_no_change": 20,    # early stopping: halt if val MAE doesn't improve for 20 rounds
        "random_state": 42,
        "n_jobs": 1,               # parallelism handled externally by joblib
        "verbose": -1,
    }

    horizon_cols = [c for c in feature_data.columns if c.startswith("h") and c[1:].isdigit()]
    mi_matrix = feature_data.set_index("feature")[horizon_cols]  # (n_features, n_horizons)
    available_features = [f for f in mi_matrix.index if f in features.columns]  # Only keep features that exist in the in-memory features DataFrame

    # Time-range filter then random row subsample
    shared_idx = features.index.intersection(future_prediction_targets_agg.index)
    shared_idx = shared_idx[(shared_idx >= subsample_start) & (shared_idx <= subsample_end)]
    n_total = len(shared_idx)
    n_rows = min(subsample_amount, n_total)
    if n_rows < n_total:
        rng = np.random.default_rng(42)
        chosen = np.sort(rng.choice(n_total, size=n_rows, replace=False))
        shared_idx = shared_idx[chosen]
    print(f"CV subsample: {n_rows:,} rows (of {n_total:,} in range {subsample_start.date()} – {subsample_end.date()})", flush=True)

    # Arrays live in the main process; threads share them without copying (no pickling, no mmap needed)
    X_arr = features.loc[shared_idx, available_features].values.astype(np.float32)  # (n_rows, n_features)
    Y_arr = future_prediction_targets_agg.loc[shared_idx].values.astype(np.float32)  # (n_rows, n_horizons)
    mi_arr = mi_matrix.loc[available_features].values.astype(np.float32)             # (n_features, n_horizons)

    x_mb = X_arr.nbytes / 1e6
    print(f"X_arr: {X_arr.shape} ({x_mb:.0f} MB) | threads: {N_PARALLEL_JOBS}", flush=True)

    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
    fold_indices = list(tscv.split(X_arr))  # pre-compute once

    # Equally-spaced grid from 1 → n_available_features, deduplicated and sorted
    k_grid = sorted(set(int(round(k)) for k in np.linspace(1, len(available_features), N_K_VALUES)))

    # Equally-spaced horizon indices to evaluate; None = all horizons
    n_h = len(horizon_cols)
    if N_HORIZON_STEPS is None or N_HORIZON_STEPS >= n_h:
        horizon_indices = list(range(n_h))
    else:
        horizon_indices = sorted(set(int(round(i)) for i in np.linspace(0, n_h - 1, N_HORIZON_STEPS)))

    n_tasks = len(horizon_indices) * len(k_grid) * len(fold_indices)
    print(
        f"Horizons: {len(horizon_indices)} of {n_h} | k grid: {k_grid} | folds: {len(fold_indices)} | "
        f"total tasks: {n_tasks}",
        flush=True,
    )

    # Per-fold worker — trains exactly one LightGBM model per call.
    # backend="threading": LightGBM fit() is pure C++ and releases the GIL, so threads get genuine
    # parallelism without spawning worker processes. Arrays are shared in the same heap — no pickling,
    # no memory-mapping, no worker heap accumulation between tasks.
    def _eval_horizon_k_fold(h_idx, k, fold_idx):
        import warnings
        warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)

        mi_scores = mi_arr[:, h_idx]
        top_k_idx = np.argsort(mi_scores)[::-1][:k]
        X_h = X_arr[:, top_k_idx]
        y = Y_arr[:, h_idx]

        train_idx, val_idx = fold_indices[fold_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        mask_tr = np.isfinite(y_tr)
        mask_va = np.isfinite(y_va)
        if mask_tr.sum() < 50 or mask_va.sum() < 10:
            return h_idx, k, fold_idx, None

        m = lgb.LGBMRegressor(**_LGBM_CV_PARAMS)
        m.fit(
            X_h[train_idx][mask_tr], y_tr[mask_tr],
            eval_set=[(X_h[val_idx][mask_va], y_va[mask_va])],
        )
        preds = m.predict(X_h[val_idx][mask_va])
        mae = float(np.mean(np.abs(y_va[mask_va] - preds)))
        del m, X_h
        return h_idx, k, fold_idx, mae

    tasks = [
        (h_idx, k, fold_idx)
        for h_idx in horizon_indices
        for k in k_grid
        for fold_idx in range(len(fold_indices))
    ]

    results_gen = Parallel(n_jobs=N_PARALLEL_JOBS, backend="threading", batch_size=1, return_as="generator_unordered")(
        delayed(_eval_horizon_k_fold)(h_idx, k, fold_idx)
        for h_idx, k, fold_idx in tasks
    )

    fold_mae_acc = {}
    for h_idx, k, fold_idx, mae in tqdm(results_gen, total=len(tasks), desc="CV feature search"):
        key = (h_idx, k)
        if key not in fold_mae_acc:
            fold_mae_acc[key] = []
        if mae is not None:
            fold_mae_acc[key].append(mae)

    cv_records = [
        {"horizon": horizon_cols[h_idx], "k": k, "cv_mae": float(np.mean(maes)) if maes else float("inf")}
        for (h_idx, k), maes in fold_mae_acc.items()
    ]

    cv_df = pd.DataFrame(cv_records)
    best_idx = cv_df.groupby("horizon")["cv_mae"].idxmin()
    horizon_best_k = cv_df.loc[best_idx].rename(columns={"k": "best_k"}).reset_index(drop=True)

    display(horizon_best_k)
    return horizon_best_k

horizon_best_k = feature_data_selection(
    feature_data_unique,
    features,
    future_prediction_targets_agg,
    subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_START"]),
    subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_END"]),
    subsample_amount=int(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
horizon_best_k.to_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_horizon_best_k_{_stem}.parquet")
)
